In [14]:
import jax
import jax.numpy as jnp
from jax import vmap
import numpy as np
import time

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "gpu")

# ============================================================
# Progress Logger
# ============================================================
class Progress:
    def __init__(self): self.t0 = time.time()
    def log(self,msg):
        t = time.time() - self.t0
        print(f"[{t:8.2f}s] {msg}")


# ============================================================
# SU(3) BASIS
# ============================================================
def su3_generators():
    lam=[]
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]],complex))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]],complex)/jnp.sqrt(3))
    return 1j * jnp.stack(lam) / 2

_T = su3_generators()


alg_from_vec = jax.jit(lambda a: jnp.einsum("...a,aij->...ij", a, _T))
su3_exp      = jax.jit(lambda A: jax.scipy.linalg.expm(A))


# ============================================================
# Lattice Structure
# ============================================================
def n_params(L): return L**4 * 4 * 8

def theta_to_field(theta, L):
    return theta.reshape(L,L,L,L,4,8)

theta_to_field = jax.jit(theta_to_field, static_argnames=('L',))


def build_links(theta, L):
    flat = theta_to_field(theta,L).reshape(-1,8)
    A = vmap(alg_from_vec)(flat)
    U = vmap(su3_exp)(A)
    return U.reshape(L,L,L,L,4,3,3)

build_links = jax.jit(build_links, static_argnames=('L',))


# ============================================================
# Wilson Action + HVP
# ============================================================
def wilson_action(theta, L, beta):
    U = build_links(theta,L)
    S=0.
    for mu in range(4):
        U1 = U[...,mu,:,:]
        for nu in range(mu+1,4):
            U2 = jnp.roll(U[...,nu,:,:], -1, axis=mu)
            U3 = jnp.conjugate(jnp.swapaxes(jnp.roll(U[...,mu,:,:],-1,axis=nu), -2,-1))
            U4 = jnp.conjugate(jnp.swapaxes(U[...,nu,:,:], -2,-1))
            P = U1 @ U2 @ U3 @ U4
            S += jnp.sum(1 - jnp.real(jnp.trace(P,axis1=-2,axis2=-1))/3)
    return beta*S

wilson_action = jax.jit(wilson_action, static_argnames=('L','beta'))
grad_S = jax.grad(wilson_action)

def hvp(theta, v, L, beta):
    _, Hv = jax.jvp(lambda x: grad_S(x,L,beta),(theta,),(v,))
    return Hv

hvp = jax.jit(hvp, static_argnames=('L','beta'))


# ============================================================
# Gauge Difference Operators
# ============================================================
def G_mul(alpha, L):
    return jnp.stack([jnp.roll(alpha,-1,axis=mu)-alpha for mu in range(4)], axis=4)

def GT_mul(v, L):
    res=jnp.zeros(v.shape[:4]+(8,))
    for mu in range(4):
        res += jnp.roll(v[...,mu,:],1,axis=mu) - v[...,mu,:]
    return res


# ============================================================
# FFT Laplacian Solver (Exact)
# ============================================================
def fft_laplace_solve(rhs, L):
    rhs_k = jnp.fft.fftn(rhs, axes=(0,1,2,3))

    k = jnp.fft.fftfreq(L)*L
    K0,K1,K2,K3 = jnp.meshgrid(k,k,k,k, indexing='ij')

    lam = 4*( jnp.sin(jnp.pi*K0/L)**2 +
              jnp.sin(jnp.pi*K1/L)**2 +
              jnp.sin(jnp.pi*K2/L)**2 +
              jnp.sin(jnp.pi*K3/L)**2 )

    lam_safe = jnp.where(lam==0,1.,lam)

    alpha_k = rhs_k / lam_safe[...,None]
    alpha_k = alpha_k.at[0,0,0,0,:].set(jnp.zeros(8))

    alpha = jnp.fft.ifftn(alpha_k, axes=(0,1,2,3))
    return jnp.real(alpha)

fft_laplace_solve = jax.jit(fft_laplace_solve, static_argnames=('L',))


# ============================================================
# Gauge Projector: P = I - G (GᵀG)^{-1} Gᵀ
# ============================================================
def build_projector(L):
    def P(v_flat):
        v = v_flat.reshape(L,L,L,L,4,8)
        rhs = GT_mul(v,L)
        alpha = fft_laplace_solve(rhs,L)
        return (v - G_mul(alpha,L)).reshape(-1)
    return jax.jit(P)


# ============================================================
# IR Basis Construction (ONLY at L=8)
# ============================================================
def build_IR_basis(L, P, extra=16):
    n = n_params(L)
    V=[]

    # Constant modes
    for mu in range(4):
        for a in range(8):
            idxs=[]; vals=[]
            for x0 in range(L):
                for x1 in range(L):
                    for x2 in range(L):
                        for x3 in range(L):
                            site=((x0*L+x1)*L+x2)*L+x3
                            idxs.append(site*32 + mu*8 + a)
                            vals.append(1.)
            v=jnp.zeros(n).at[jnp.array(idxs)].set(jnp.array(vals))
            v=P(v)
            if jnp.linalg.norm(v)>1e-12: V.append(v/jnp.linalg.norm(v))

    # Momentum-1 modes
    for mu in range(4):
        for a in range(8):
            for trig in ["cos","sin"]:
                fun=jnp.cos if trig=="cos" else jnp.sin
                idxs=[]; vals=[]
                for x0 in range(L):
                    for x1 in range(L):
                        for x2 in range(L):
                            for x3 in range(L):
                                coord=[x0,x1,x2,x3][mu]
                                vals.append(fun(2*jnp.pi*coord/L))
                                site=((x0*L+x1)*L+x2)*L+x3
                                idxs.append(site*32 + mu*8 + a)
                v=jnp.zeros(n).at[jnp.array(idxs)].set(jnp.array(vals))
                v=P(v)
                if jnp.linalg.norm(v)>1e-12: V.append(v/jnp.linalg.norm(v))

    # Random block modes
    for k in range(extra):
        v=jax.random.normal(jax.random.PRNGKey(k),(n,))
        v=P(v)
        if jnp.linalg.norm(v)>1e-12: V.append(v/jnp.linalg.norm(v))

    V=jnp.stack(V)
    Q,_=jnp.linalg.qr(V.T)
    return Q.T   # shape (m,n)


# ============================================================
# Batched Hessian
# ============================================================
def build_Hsub(theta, V, L, beta):
    HV = vmap(lambda vi: hvp(theta,vi,L,beta))(V)
    H  = V @ HV.T
    return 0.5*(H+H.T)


# ============================================================
# Block Map J
# ============================================================
def build_J(L):
    nf=n_params(L)
    Lc=L//2
    nc=n_params(Lc)
    J=jnp.zeros((nf,nc))

    for mu in range(4):
        for a in range(8):
            cidx = mu*8 + a
            fine = [
                (0,0,0,0),
                (
                    1 if mu==0 else 0,
                    1 if mu==1 else 0,
                    1 if mu==2 else 0,
                    1 if mu==3 else 0
                )
            ]
            for (x0,x1,x2,x3) in fine:
                s=((x0*L+x1)*L+x2)*L+x3
                J = J.at[s*32+mu*8+a,cidx].set(0.5)
    return J


# ============================================================
# Riccati + Curvature
# ============================================================
def riccati_small(H, steps=12, eta=0.05):
    I=jnp.eye(H.shape[0])
    Hn=H
    for _ in range(steps):
        Hn = Hn @ jnp.linalg.inv(I+eta*Hn)
    return 0.5*(Hn+Hn.T)

def lambda_min(H,tol=1e-6):
    w=jnp.linalg.eigvalsh(H)
    w=w[jnp.abs(w)>tol]
    return float(w.min()) if w.size>0 else 0.


# ============================================================
# FINAL CLEANED RG DRIVER
# ============================================================
def curvature_RG_L8(beta=6.0, extra=16, steps=3):

    prog = Progress()
    L=8
    a=1.0
    theta0=jnp.zeros(n_params(L))

    prog.log("Building IR basis at L=8 (single basis)")
    P8 = build_projector(8)
    V = build_IR_basis(8, P8, extra=extra)
    m = V.shape[0]
    prog.log(f"Initial IR basis dimension: {m}")

    scales=[]
    curvs=[]

    for step in range(steps+1):

        prog.log(f"=== RG SCALE a={a:.3f}  L={L}  step={step} ===")

        # Build projector for this L
        P = build_projector(L)

        # If L=1 → terminate (no physical modes remain)
        if L == 1:
            prog.log("Reached L=1: terminating RG.")
            break

        # Compute Hessian
        prog.log("Building Hessian")
        Hsub = build_Hsub(theta0,V,L,beta)

        prog.log("Riccati")
        Hren = riccati_small(Hsub)

        CWr = lambda_min(Hren)
        Meff = CWr / a

        prog.log(f"CWr={CWr}, Meff={Meff}\n")
        scales.append((L,a))
        curvs.append((CWr,Meff))

        if L == 1:
            break

        # ----- COARSE STEP -----
        prog.log("Coarse RG step")

        Lc=L//2
        Jf=build_J(L)
        Pc = build_projector(Lc)

        prog.log("Batched coarse mapping of IR basis")

        # JV_all: (m,nf) × (nf,nc) → (m,nc)
        JV_all = vmap(lambda vi: Jf.T @ vi)(V)

        # Coarse projector (batched)
        Vc_raw = vmap(Pc)(JV_all)

        # Orthonormalize coarse basis (same m)
        Qc,_ = jnp.linalg.qr(Vc_raw.T)
        V = Qc.T  # updated IR basis for next scale

        # Build coarse Hessian mapping Jsub
        Jsub = V @ JV_all.T

        # Update Hessian for next round
        Hsub = 0.5*(Jsub.T @ Hren @ Jsub + (Jsub.T @ Hren @ Jsub).T)

        theta0 = jnp.zeros(n_params(Lc))
        L=Lc
        a=2*a

    # ===== Summary =====
    print("\n========================================")
    print(" CURVATURE RG FLOW SUMMARY")
    print("========================================")
    print(f"{'L':>4} {'a':>8} {'CWr':>20} {'M_eff':>20}")
    for (L,a),(cwr,meff) in zip(scales,curvs):
        print(f"{L:>4} {a:>8.3f} {cwr:>20.12e} {meff:>20.12e}")
    print("========================================\n")


# ============================================================
# RUN
# ============================================================
curvature_RG_L8(beta=6.0, extra=16, steps=3)


[    0.00s] Building IR basis at L=8 (single basis)
[   96.88s] Initial IR basis dimension: 48
[   96.88s] === RG SCALE a=1.000  L=8  step=0 ===
[   96.88s] Building Hessian
[  107.12s] Riccati
[  107.89s] CWr=1.3773161794003694, Meff=1.3773161794003694

[  107.89s] Coarse RG step
[  108.56s] Batched coarse mapping of IR basis
[  109.42s] === RG SCALE a=2.000  L=4  step=1 ===
[  109.42s] Building Hessian
[  119.49s] Riccati
[  119.58s] CWr=0.04935635592477897, Meff=0.024678177962389487

[  119.58s] Coarse RG step
[  119.68s] Batched coarse mapping of IR basis
[  120.46s] === RG SCALE a=4.000  L=2  step=2 ===
[  120.47s] Building Hessian
[  127.85s] Riccati
[  127.86s] CWr=0.08628853897440678, Meff=0.021572134743601694

[  127.86s] Coarse RG step
[  127.99s] Batched coarse mapping of IR basis


TypeError: dot_general requires contracting dimensions to have the same shape, got (32,) and (48,).

In [16]:
import jax
import jax.numpy as jnp
from jax import vmap
import numpy as np
import time

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "gpu")


# ============================================================
# Progress logger
# ============================================================
class Progress:
    def __init__(self):
        self.t0 = time.time()
    def log(self, msg):
        dt = time.time() - self.t0
        print(f"[{dt:8.2f}s] {msg}")


# ============================================================
# SU(3) generators
# ============================================================
def su3_generators():
    lam=[]
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]],complex))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]],complex))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]],complex))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]],complex))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]],complex)/jnp.sqrt(3))
    return 1j * jnp.stack(lam) / 2

_T = su3_generators()


def alg_from_vec(a):
    return jnp.einsum("...a,aij->...ij", a, _T)
alg_from_vec = jax.jit(alg_from_vec)


def su3_exp(A):
    return jax.scipy.linalg.expm(A)
su3_exp = jax.jit(su3_exp)


# ============================================================
# Lattice structure
# ============================================================
def n_params(L):
    return L**4 * 4 * 8


def theta_to_field(theta, L):
    return theta.reshape(L, L, L, L, 4, 8)
theta_to_field = jax.jit(theta_to_field, static_argnames=('L',))


def build_links(theta, L):
    flat = theta_to_field(theta, L).reshape(-1, 8)
    A = vmap(alg_from_vec)(flat)
    U = vmap(su3_exp)(A)
    return U.reshape(L, L, L, L, 4, 3, 3)
build_links = jax.jit(build_links, static_argnames=('L',))


# ============================================================
# Wilson action + right-invariant HVP
# ============================================================
def wilson_action(theta, L, beta):
    U = build_links(theta, L)
    S = 0.0
    for mu in range(4):
        U1 = U[...,mu,:,:]
        for nu in range(mu+1,4):
            U2 = jnp.roll(U[...,nu,:,:], -1, axis=mu)
            U3 = jnp.conjugate(jnp.swapaxes(jnp.roll(U[...,mu,:,:],-1,axis=nu), -1,-2))
            U4 = jnp.conjugate(jnp.swapaxes(U[...,nu,:,:], -1,-2))
            P = U1 @ U2 @ U3 @ U4
            S += jnp.sum(1 - jnp.real(jnp.trace(P, axis1=-2, axis2=-1))/3)
    return beta * S
wilson_action = jax.jit(wilson_action, static_argnames=('L','beta'))


grad_S = jax.grad(wilson_action)


def hvp(theta, v, L, beta):
    _, Hv = jax.jvp(lambda x: grad_S(x,L,beta), (theta,), (v,))
    return Hv
hvp = jax.jit(hvp, static_argnames=('L','beta'))


# ============================================================
# Gauge operator stencils (sparse)
# ============================================================
def G_mul(alpha, L):
    v=[]
    for mu in range(4):
        fwd = jnp.roll(alpha, shift=-1, axis=mu)
        v.append(fwd - alpha)
    return jnp.stack(v, axis=4)


def GT_mul(v, L):
    res=jnp.zeros(v.shape[:4] + (8,))
    for mu in range(4):
        back = jnp.roll(v[...,mu,:], shift=1, axis=mu)
        res = res + (back - v[...,mu,:])
    return res


def laplace_mul(alpha, L):
    return GT_mul(G_mul(alpha, L), L)


# ============================================================
# Sparse CG solver
# ============================================================
def cg_solve(A_mul, b, maxiter=600, tol=1e-12):
    x=jnp.zeros_like(b)
    r=b - A_mul(x)
    p=r
    rs=jnp.vdot(r,r)

    def body(carry, k):
        x, r, p, rs = carry
        Ap = A_mul(p)
        alpha = rs / jnp.vdot(p,Ap)
        x2 = x + alpha*p
        r2 = r - alpha*Ap
        rs2 = jnp.vdot(r2,r2)
        beta = rs2 / rs
        p2 = r2 + beta*p
        return (x2, r2, p2, rs2), None

    (xf, _, _, _), _ = jax.lax.scan(body, (x,r,p,rs), jnp.arange(maxiter))
    return xf


# ============================================================
# Projector P(v) = v - G (GᵀG)⁻¹ Gᵀ v
# ============================================================
def build_projector(L):

    def P_inner(v_flat, L_static):
        v = v_flat.reshape(L_static,L_static,L_static,L_static,4,8)
        rhs = GT_mul(v, L_static)
        alpha = cg_solve(lambda a: laplace_mul(a,L_static), rhs)
        out = v - G_mul(alpha, L_static)
        return out.reshape(-1)
    P_jitted = jax.jit(P_inner, static_argnames=('L_static',))
    return lambda v: P_jitted(v, L_static=L)


# ============================================================
# IR basis (indexing fixed)
# ============================================================
def build_IR_basis(L, P, extra=16, prog=None):
    n=n_params(L)
    V=[]

    # constant modes
    for mu in range(4):
        for a in range(8):
            if prog: prog.log(f"IR const μ={mu} a={a}")
            idxs=[]
            vals=[]
            for x0 in range(L):
                for x1 in range(L):
                    for x2 in range(L):
                        for x3 in range(L):
                            s=((x0*L+x1)*L+x2)*L+x3
                            idxs.append(s*32 + mu*8 + a)
                            vals.append(1.0)
            idxs=jnp.array(idxs, dtype=jnp.int32)
            vals=jnp.array(vals, dtype=jnp.float64)
            v=jnp.zeros(n).at[idxs].set(vals)
            v=P(v)
            nrm=jnp.linalg.norm(v)
            if nrm>1e-12: V.append(v/nrm)

    # momentum modes
    for mu in range(4):
        for a in range(8):
            if prog: prog.log(f"IR k=1 μ={mu} a={a}")
            for trig in ["cos","sin"]:
                fun = jnp.cos if trig=="cos" else jnp.sin
                idxs=[]; vals=[]
                for x0 in range(L):
                    for x1 in range(L):
                        for x2 in range(L):
                            for x3 in range(L):
                                coord=[x0,x1,x2,x3][mu]
                                phase=fun(2*jnp.pi*coord/L)
                                site=((x0*L+x1)*L+x2)*L+x3
                                idxs.append(site*32 + mu*8 + a)
                                vals.append(phase)
                idxs=jnp.array(idxs, dtype=jnp.int32)
                vals=jnp.array(vals, dtype=jnp.float64)
                v=jnp.zeros(n).at[idxs].set(vals)
                v=P(v)
                nrm=jnp.linalg.norm(v)
                if nrm>1e-12: V.append(v/nrm)

    # random block modes
    for k in range(extra):
        if prog: prog.log(f"IR random {k+1}/{extra}")
        v=jax.random.normal(jax.random.PRNGKey(k),(n,))
        v=P(v)
        nrm=jnp.linalg.norm(v)
        if nrm>1e-12: V.append(v/nrm)

    # Handle case where V is empty
    if not V:
        return jnp.empty((0, n)) # Return an empty array with correct number of columns

    V=jnp.stack(V)
    Q,_ = jnp.linalg.qr(V.T)
    return Q.T


# ============================================================
# Subspace Hessian
# ============================================================
def build_Hsub(theta, V, L, beta=1.0, prog=None):
    m,n=V.shape
    rows=[]
    for i in range(m):
        if prog and i%5==0:
            prog.log(f"HVP row {i}/{m}")
        Hv = hvp(theta, V[i], L, beta)
        rows.append(jnp.dot(V,Hv))
    H=jnp.stack(rows)
    return 0.5*(H + H.T)


# ============================================================
# Block RG map
# ============================================================
def build_J(L):
    nf=n_params(L)
    Lc=L//2
    nc=n_params(Lc)
    J=np.zeros((nf,nc))
    for mu in range(4):
        for a in range(8):
            cidx=mu*8 + a
            fine=[
                (0,0,0,0),
                (1 if mu==0 else 0,
                 1 if mu==1 else 0,
                 1 if mu==2 else 0,
                 1 if mu==3 else 0)
            ]
            for (x0,x1,x2,x3) in fine:
                s=((x0*L+x1)*L+x2)*L+x3
                J[s*32 + mu*8 + a, cidx]=0.5
    return jnp.array(J)


# ============================================================
# Coarse Hessian
# ============================================================
def coarse_Hsub(H, Jsub):
    # Jsub is (m_coarse, m_fine)
    # H is (m_fine, m_fine)
    # Jsub.T is (m_fine, m_coarse)
    # Result should be (m_coarse, m_coarse)
    M = Jsub @ H @ Jsub.T # Corrected multiplication order
    return 0.5*(M + M.T)


# ============================================================
# Riccati + curvature
# ============================================================
def riccati_small(H, steps=12, eta=0.05):
    I=jnp.eye(H.shape[0])
    Hn=H
    for _ in range(steps):
        Hn = Hn @ jnp.linalg.inv(I + eta*Hn)
    return 0.5*(Hn + Hn.T)


def lambda_min(H, tol=1e-6):
    w=jnp.linalg.eigvalsh(H)
    w=w[jnp.abs(w)>tol]
    return float(w.min()) if w.size>0 else 0.0


# ============================================================
# MAIN RG DRIVER
# ============================================================
def curvature_RG_L8(beta=6.0, extra=16, steps=3):
    prog = Progress()

    L=8
    a=1.0
    theta0=jnp.zeros(n_params(L))

    for step in range(steps+1):

        prog.log(f"=== RG SCALE a={a:.3f} L={L} step={step} ===")

        # Projector
        P = build_projector(L)

        # IR basis
        prog.log("Building IR basis")
        V = build_IR_basis(L, P, extra, prog=prog)
        m_fine = V.shape[0] # Dimension of the current fine basis

        # If the IR basis for the current L is empty, terminate
        if m_fine == 0:
            prog.log(f"IR basis for L={L} is empty. Terminating RG.")
            break

        # Hessian
        prog.log("Building subspace Hessian")
        Hsub = build_Hsub(theta0, V, L, beta, prog=prog)

        # Riccati
        prog.log("Riccati flow")
        Hren = riccati_small(Hsub)

        # Curvature
        prog.log("Curvature")
        CWr = lambda_min(Hren)
        print("CWr =", CWr, "M_eff =", CWr/a)

        # If L is 1, this is the last step with meaningful curvature, break before coarse-graining further.
        if L == 1:
            break

        # Coarse step
        prog.log("Coarse RG")
        Lc=L//2
        Pc = build_projector(Lc)
        Vc = build_IR_basis(Lc, Pc, extra, prog=prog)
        m_coarse = Vc.shape[0] # Dimension of the coarse basis

        # If the coarse basis is empty, terminate
        if m_coarse == 0:
            prog.log(f"IR basis for L={Lc} is empty. Terminating RG.")
            break

        Jf = build_J(L)

        # Initialize Jsub with correct dimensions (m_coarse, m_fine)
        Jsub=jnp.zeros((m_coarse, m_fine))

        # Populate Jsub
        for i in range(m_fine):
            if prog and i%5==0: prog.log(f"Project mode {i}/{m_fine}")
            JV = Pc(Jf.T @ V[i])
            # Vc.shape is (m_coarse, n). JV.shape is (n,)
            # We want to project JV onto the Vc basis. This means finding coefficients 'c' such that JV ≈ Vc.T @ c
            # Or, finding the component of JV along each Vc basis vector. jnp.dot(Vc, JV) does exactly that.
            # The result is a vector of length m_coarse.
            Jsub = Jsub.at[:,i].set(jnp.dot(Vc, JV))

        Hsub = coarse_Hsub(Hren, Jsub)

        # Prep next
        theta0=jnp.zeros(n_params(Lc))
        L=Lc
        a=2*a


# ============================================================
# RUN
# ============================================================
curvature_RG_L8(beta=6.0, extra=16, steps=3)


[    0.00s] === RG SCALE a=1.000 L=8 step=0 ===
[    0.00s] Building IR basis
[    0.00s] IR const μ=0 a=0
[    0.52s] IR const μ=0 a=1
[    0.54s] IR const μ=0 a=2
[    0.56s] IR const μ=0 a=3
[    0.58s] IR const μ=0 a=4
[    0.61s] IR const μ=0 a=5
[    0.63s] IR const μ=0 a=6
[    0.65s] IR const μ=0 a=7
[    0.67s] IR const μ=1 a=0
[    0.69s] IR const μ=1 a=1
[    0.72s] IR const μ=1 a=2
[    0.74s] IR const μ=1 a=3
[    0.76s] IR const μ=1 a=4
[    0.78s] IR const μ=1 a=5
[    0.80s] IR const μ=1 a=6
[    0.83s] IR const μ=1 a=7
[    0.85s] IR const μ=2 a=0
[    0.86s] IR const μ=2 a=1
[    0.88s] IR const μ=2 a=2
[    0.90s] IR const μ=2 a=3
[    0.92s] IR const μ=2 a=4
[    0.94s] IR const μ=2 a=5
[    0.96s] IR const μ=2 a=6
[    0.98s] IR const μ=2 a=7
[    1.00s] IR const μ=3 a=0
[    1.02s] IR const μ=3 a=1
[    1.04s] IR const μ=3 a=2
[    1.06s] IR const μ=3 a=3
[    1.08s] IR const μ=3 a=4
[    1.10s] IR const μ=3 a=5
[    1.12s] IR const μ=3 a=6
[    1.13s] IR const μ=

/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


[  101.42s] HVP row 5/16
[  101.63s] HVP row 10/16
[  101.83s] HVP row 15/16
[  101.88s] Riccati flow
[  102.00s] Curvature
CWr = 0.002168349332698308 M_eff = 0.002168349332698308
[  102.00s] Coarse RG
[  102.00s] IR const μ=0 a=0
[  102.52s] IR const μ=0 a=1
[  102.53s] IR const μ=0 a=2
[  102.54s] IR const μ=0 a=3
[  102.55s] IR const μ=0 a=4
[  102.56s] IR const μ=0 a=5
[  102.58s] IR const μ=0 a=6
[  102.59s] IR const μ=0 a=7
[  102.60s] IR const μ=1 a=0
[  102.61s] IR const μ=1 a=1
[  102.62s] IR const μ=1 a=2
[  102.63s] IR const μ=1 a=3
[  102.65s] IR const μ=1 a=4
[  102.66s] IR const μ=1 a=5
[  102.67s] IR const μ=1 a=6
[  102.68s] IR const μ=1 a=7
[  102.69s] IR const μ=2 a=0
[  102.70s] IR const μ=2 a=1
[  102.72s] IR const μ=2 a=2
[  102.73s] IR const μ=2 a=3
[  102.74s] IR const μ=2 a=4
[  102.75s] IR const μ=2 a=5
[  102.76s] IR const μ=2 a=6
[  102.77s] IR const μ=2 a=7
[  102.78s] IR const μ=3 a=0
[  102.80s] IR const μ=3 a=1
[  102.81s] IR const μ=3 a=2
[  102.82s] IR 

In [17]:
    return scales, curvs


SyntaxError: 'return' outside function (ipython-input-707049415.py, line 1)